# Research Version 3 ? 01: Dataset provenance and suitability

Inspect the acquisition manifests and schemas. DataCo supplies historical order outcomes; India-Nepal and CrisisMMD provide distinct incident-label tasks. Dataset coexistence does not establish shipment exposure.

This notebook runs only in Colab and Google Drive. No Windows checkout, local datasets, local model files or replication-manifest file are required. Use the new experiment ID; do not mix this package with an older cpu-v3 archive.


# 01: Versioned acquisition and provenance

## 1. Purpose

Acquire DataCo, India-Nepal and text-only CrisisMMD from pinned public sources.

## 2. Prerequisites and outputs

**Required inputs:** Notebook 00 environment and shared controller.

**This stage saves:** 18 intended raw files and three provenance/integrity manifests.

Required earlier notebook markers: 00.

All runtime files belong to this Colab/Drive experiment; no external project files are read.

## 3. Configuration

Use the same experiment ID/folder throughout. All action switches default False. Enable Drive/bootstrap and RUN_STAGE deliberately; enable installation on a fresh runtime. Notebook 01 needs dataset download permission; notebook 08 needs pretrained download permission.

In [ ]:
EXPERIMENT_ID = "colab-cpu-v3-clean"
DRIVE_FOLDER = "ChainShield-Colab"
ENABLE_DRIVE = False
BOOTSTRAP_EXPERIMENT = False
ALLOW_PACKAGE_INSTALL = False
ALLOW_DATA_DOWNLOADS = False
ALLOW_PRETRAINED_DOWNLOADS = False
RUN_STAGE = False
SYNC_INTERVAL_SECONDS = 60
VERIFY_PERSISTENCE_FULL = False
VERIFY_CHECKPOINT_PAIRS = False
CLEAR_STALE_WRITER_LOCK = False
CONFIRM_NO_OTHER_WRITER = False
DOWNLOAD_REPORT_ZIP = False

# Isolated plotting workers must not inherit the Colab inline backend.
if RUN_STAGE:
    import os
    os.environ["MPLBACKEND"] = "Agg"


## 4. Readable stage implementation

The small cell command below registers visible module text in memory. It does not execute the module. This notebook supplies only the files listed here; other supporting modules are restored from earlier Colab stages.

In [ ]:
# This small cell registers the visible stage code; it writes no files.
REGISTERED_FILES = {}
def register_stage_file(line, body):
    name=line.strip()
    if not name or name.startswith("/") or "\\" in name or ":" in name or ".." in name.split("/"):
        raise ValueError("Use a safe relative module name")
    REGISTERED_FILES[name]=body.rstrip("\n")+"\n"
get_ipython().register_magic_function(register_stage_file,"cell","chainshield_file")


### configs/datasets.json


In [ ]:
%%chainshield_file configs/datasets.json
{
  "schema_version": 1,
  "status": "configured_sources_not_yet_acquired",
  "acquisition": {
    "automatic_download": false,
    "require_explicit_approval": true,
    "raw_root": "data/raw",
    "manifest_root": "data/manifests",
    "checksum": "sha256",
    "preserve_original_files": true
  },
  "datasets": [
    {
      "id": "dataco",
      "name": "DataCo Smart Supply Chain",
      "url": "https://data.mendeley.com/datasets/8gx2fvg2k6/5",
      "proposed_version": "5",
      "destination": "data/raw/dataco",
      "required_inputs": [
        "Structured order-item CSV",
        "Field descriptions",
        "Source and license metadata"
      ],
      "target": "Late_delivery_risk",
      "unit_of_analysis": "order",
      "prediction_cutoff": "order_creation_provisional",
      "split_policy": "Chronological order-level train/validation/test; fit preprocessing on train only",
      "audit_requirements": [
        "Order-ID consistency",
        "Conflicting targets within an order",
        "Post-outcome leakage",
        "Shipping-mode-only baseline"
      ],
      "license_status": "verify_before_acquisition"
    },
    {
      "id": "india_nepal",
      "name": "India-Nepal Disaster Tweet Corpus",
      "url": "https://data.mendeley.com/datasets/psv6b4zy9f/1",
      "proposed_version": "1",
      "destination": "data/raw/india_nepal",
      "required_inputs": [
        "Corpus files",
        "Actual label hierarchy and annotation definitions",
        "Source and license metadata"
      ],
      "proposed_tasks": [
        "disaster_relevance_subject_to_label_audit",
        "flood_vs_earthquake_subject_to_label_audit"
      ],
      "split_policy": "Preserve source splits where valid; audit exact and near duplicates and event overlap",
      "license_status": "verify_before_acquisition"
    },
    {
      "id": "crisismmd",
      "name": "CrisisMMD text only",
      "url": "https://huggingface.co/datasets/QCRI/CrisisMMD",
      "requested_revision": "10a5626ba112ab9c50369c473a2062bcf205d708",
      "resolved_revision": null,
      "destination": "data/raw/crisismmd",
      "required_inputs": [
        "informative/{train,dev,test}.json",
        "humanitarian/{train,dev,test}.json",
        "files_individual_events/*.tsv",
        "README and license metadata"
      ],
      "exclude": [
        "Images",
        "Image-grounded damage-severity training targets"
      ],
      "proposed_tasks": [
        "text_informativeness",
        "text_humanitarian_information"
      ],
      "split_policy": "Preserve task provenance; export task-specific and shared splits; quarantine cross-split duplicate text and unresolved labels",
      "license_status": "notebook_reports_CC_BY_NC_SA_4_0_verify_at_pinned_revision",
      "revision_policy": "Acquire this immutable revision and record source checksums"
    }
  ]
}


### scripts/acquire_colab_datasets.py


In [ ]:
%%chainshield_file scripts/acquire_colab_datasets.py
"""Download only the explicitly selected ChainShield research inputs.

Uses the standard library, pins source versions, verifies upstream integrity,
and refuses to overwrite existing datasets, partial files, or manifests.
No archive extraction, dependency installation, training, or automatic retry.
"""
from __future__ import annotations

import argparse
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
from urllib.parse import quote, urlsplit, urlunsplit
from urllib.request import Request, urlopen

ROOT = Path(__file__).resolve().parents[1]
REVISION = "10a5626ba112ab9c50369c473a2062bcf205d708"
USER_AGENT = "ChainShield-Academic-Capstone/0.1"
EVENTS = (
    "california_wildfires", "hurricane_harvey", "hurricane_irma",
    "hurricane_maria", "iraq_iran_earthquake", "mexico_earthquake",
    "srilanka_floods",
)
MENDELEY = {
    "dataco": {
        "dataset": "8gx2fvg2k6", "version": 5,
        "files": {
            "DataCoSupplyChainDataset.csv": (
                95910149,
                "fa6d022ed437155e1a2f0378710602848703c8a7f203f7ff5d77805bf8480aa6",
            ),
            "DescriptionDataCoSupplyChain.csv": (
                3444,
                "9828e34669bd6d77e3b4463364cc44a5d52446b5e246fc258758cfe592566c4b",
            ),
        },
    },
    "india_nepal": {
        "dataset": "psv6b4zy9f", "version": 1,
        "files": {
            "dataset.zip": (
                2618917,
                "17f7ee65d3f7b1151cc4affdc1527d5d598725ba01b3d0f955c5440dd39f501d",
            ),
        },
    },
}


def now():
    return datetime.now(timezone.utc).isoformat()


def metadata(url):
    """Read bounded JSON metadata into memory, without caching to disk."""
    with urlopen(Request(url, headers={"User-Agent": USER_AGENT}), timeout=30) as response:
        data = response.read(2_000_001)
    if len(data) > 2_000_000:
        raise ValueError("Unexpectedly large metadata response")
    return json.loads(data.decode("utf-8"))


def within_root(relative):
    target = (ROOT / relative).resolve()
    target.relative_to(ROOT)
    return target


def public_url(url):
    """Avoid persisting signed CDN query parameters."""
    parts = urlsplit(url)
    return urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))


def mendeley_plan(key):
    spec = MENDELEY[key]
    dataset, version = spec["dataset"], spec["version"]
    listing = metadata(
        f"https://data.mendeley.com/public-api/datasets/{dataset}"
        f"/files?folder_id=root&version={version}"
    )
    if not isinstance(listing, list):
        raise ValueError("Expected a Mendeley file listing")
    plan = []
    for filename, (size, checksum) in spec["files"].items():
        matches = [item for item in listing if item.get("filename") == filename]
        if len(matches) != 1:
            raise ValueError(f"Expected exactly one source file: {filename}")
        details = matches[0]["content_details"]
        if details["size"] != size or details["sha256_hash"] != checksum:
            raise ValueError(f"Source metadata changed: {filename}")
        url = details["download_url"]
        if urlsplit(url).scheme != "https" or urlsplit(url).hostname != "data.mendeley.com":
            raise ValueError("Unexpected Mendeley download URL")
        plan.append({
            "relative_path": f"data/raw/{key}/{filename}",
            "url": url, "expected_bytes": size,
            "upstream_sha256": checksum, "upstream_git_blob_sha1": None,
        })
    return {
        "dataset_id": key,
        "source_url": f"https://data.mendeley.com/datasets/{dataset}/{version}",
        "source_revision": str(version),
        "published_license": "CC BY 4.0",
        "license_basis": "Published dataset landing page verified during access review",
        "files": plan,
    }


def crisismmd_plan():
    api = f"https://huggingface.co/api/datasets/QCRI/CrisisMMD/tree/{REVISION}"
    desired = {"README.md", "licenses_by-nc-sa_4.0_legalcode.txt"}
    desired.update(f"{task}/{split}.json"
                   for task in ("informative", "humanitarian")
                   for split in ("train", "dev", "test"))
    desired.update(f"files_individual_events/{event}_final_data.tsv" for event in EVENTS)
    entries = []
    for directory in ("", "informative", "humanitarian", "files_individual_events"):
        url = api + ("/" + directory if directory else "")
        listing = metadata(url)
        if not isinstance(listing, list):
            raise ValueError("Expected a Hugging Face file listing")
        entries.extend(item for item in listing
                       if item.get("type") == "file" and item.get("path") in desired)
    by_path = {item["path"]: item for item in entries}
    if set(by_path) != desired or len(entries) != len(desired):
        raise ValueError("Pinned CrisisMMD snapshot is missing or duplicating selected files")
    plan = []
    for name in sorted(desired):
        item = by_path[name]
        size = item["size"]
        if not isinstance(size, int) or not 0 < size < 10_000_000:
            raise ValueError(f"Unexpected CrisisMMD file size: {name}")
        lfs = item.get("lfs")
        sha256 = lfs["oid"] if lfs else None
        git_blob = None if lfs else item["oid"]
        if lfs and lfs["size"] != size:
            raise ValueError("Inconsistent LFS file size")
        plan.append({
            "relative_path": "data/raw/crisismmd/" + name,
            "url": f"https://huggingface.co/datasets/QCRI/CrisisMMD/resolve/"
                   f"{REVISION}/{quote(name, safe='/')}",
            "expected_bytes": size,
            "upstream_sha256": sha256,
            "upstream_git_blob_sha1": git_blob,
        })
    return {
        "dataset_id": "crisismmd",
        "source_url": "https://huggingface.co/datasets/QCRI/CrisisMMD",
        "source_revision": REVISION,
        "published_license": "CC BY-NC-SA 4.0",
        "license_basis": "Repository metadata and included license text; retain source restrictions",
        "excluded": ["data_image/", "damage/"],
        "files": plan,
    }


def preflight(plans):
    """Check every destination before making any local changes."""
    targets = []
    for plan in plans:
        targets.append(within_root(f"data/manifests/{plan['dataset_id']}.json"))
        for item in plan["files"]:
            target = within_root(item["relative_path"])
            targets.extend((target, target.with_name(target.name + ".partial")))
    if len(set(targets)) != len(targets):
        raise ValueError("Duplicate destinations in acquisition plan")
    for target in targets:
        if target.exists():
            raise FileExistsError(f"Refusing to overwrite: {target.relative_to(ROOT)}")


def download(item):
    target = within_root(item["relative_path"])
    partial = target.with_name(target.name + ".partial")
    target.parent.mkdir(parents=True, exist_ok=True)
    expected = item["expected_bytes"]
    sha256 = hashlib.sha256()
    blob = hashlib.sha1()
    blob.update(f"blob {expected}\0".encode("ascii"))
    started = now()
    count = 0
    progress_mark = 0
    print(f"Downloading {item['relative_path']} ({expected:,} bytes)", flush=True)
    request = Request(item["url"], headers={
        "User-Agent": USER_AGENT, "Accept-Encoding": "identity",
    })
    # Exclusive creation protects existing data. A failed partial is retained
    # for inspection; reruns stop rather than delete or silently replace it.
    with urlopen(request, timeout=60) as response, partial.open("xb") as handle:
        if response.status != 200:
            raise ValueError(f"Unexpected download status: {response.status}")
        length = response.headers.get("Content-Length")
        if length is not None and int(length) != expected:
            raise ValueError("HTTP Content-Length differs from source metadata")
        while chunk := response.read(1024 * 1024):
            count += len(chunk)
            if count > expected:
                raise ValueError("Transfer exceeds expected source size")
            handle.write(chunk)
            sha256.update(chunk)
            blob.update(chunk)
            if count - progress_mark >= 8 * 1024 * 1024:
                print(f"  {count:,} / {expected:,} bytes", flush=True)
                progress_mark = count
        final_source = public_url(response.url)
    if count != expected:
        raise ValueError(f"Incomplete transfer: expected {expected}, received {count}")
    digest = sha256.hexdigest()
    if item["upstream_sha256"] and digest != item["upstream_sha256"]:
        raise ValueError("SHA-256 differs from the upstream checksum")
    if item["upstream_git_blob_sha1"] and blob.hexdigest() != item["upstream_git_blob_sha1"]:
        raise ValueError("Git blob digest differs from the pinned repository")
    # Recheck before publishing a downloaded file; never replace an existing input.
    if target.exists():
        raise FileExistsError("Destination appeared during download")
    partial.rename(target)
    print(f"Verified {item['relative_path']}", flush=True)
    return {
        "file": item["relative_path"],
        "bytes": count, "sha256": digest,
        "upstream_sha256": item["upstream_sha256"],
        "upstream_git_blob_sha1": item["upstream_git_blob_sha1"],
        "integrity_check": "sha256" if item["upstream_sha256"] else "git_blob_sha1",
        "source_url": item["url"],
        "resolved_download_url_without_query": final_source,
        "acquired_at_utc": started,
        "verified_at_utc": now(),
    }


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dataset", required=True,
                        choices=("all", "dataco", "india_nepal", "crisismmd"))
    args = parser.parse_args()
    if not (ROOT / "configs/datasets.json").is_file():
        raise ValueError("Register notebook 01's dataset configuration first")
    selected = ["dataco", "india_nepal", "crisismmd"] if args.dataset == "all" else [args.dataset]
    plans = [crisismmd_plan() if key == "crisismmd" else mendeley_plan(key)
             for key in selected]
    preflight(plans)
    total = sum(item["expected_bytes"] for plan in plans for item in plan["files"])
    print(f"Selected {sum(len(p['files']) for p in plans)} files; {total:,} bytes", flush=True)
    for plan in plans:
        results = [download(item) for item in plan["files"]]
        manifest = {key: value for key, value in plan.items() if key != "files"}
        manifest.update({
            "schema_version": 1, "status": "downloaded_and_integrity_verified",
            "created_at_utc": now(), "files": results,
            "file_count": len(results),
            "total_bytes": sum(item["bytes"] for item in results),
            "archive_extracted": False if plan["dataset_id"] == "india_nepal" else None,
            "limitations": [
                "Integrity verification does not validate labels, licensing interpretations, or model suitability.",
                "No preprocessing, model training, or archive extraction performed.",
            ],
        })
        target = within_root(f"data/manifests/{plan['dataset_id']}.json")
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("x", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2, ensure_ascii=False)
            handle.write("\n")
        print(f"Manifest saved: {target.relative_to(ROOT)}", flush=True)
    print("Acquisition complete; all selected files passed integrity checks.", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import hashlib,json
EXPECTED_STAGE_FILES={'configs/datasets.json': '5f020bc0278667c2a38eff98b8f395833d19dcd664fca5b7cffe5175c8cf6a4f', 'scripts/acquire_colab_datasets.py': '3cc038454b6b5113ced9052caf65a043cbbb3d5bdca2f47fc85ebee12d2b5879'}
assert set(REGISTERED_FILES)==set(EXPECTED_STAGE_FILES)
for name,expected in EXPECTED_STAGE_FILES.items():
    if hashlib.sha256(REGISTERED_FILES[name].encode()).hexdigest()!=expected:raise ValueError("Changed source: "+name)


## 5. Attach, verify prerequisites and execute

Bootstrap connects Drive and restores committed work. Source installation commits only this stage definitions. Dependencies must already have completed; this notebook does not silently train them.

In [ ]:
# Connect the Colab experiment and load the shared runtime.
from pathlib import Path
CONTROLLER_SHA = '8c092c59baffc0804aeca86b4f0ac3fb6d4be3cd96eea96c9e7299e559d9c9e6'
IMPLEMENTATION_SHA256 = CONTROLLER_SHA
rt = None
if BOOTSTRAP_EXPERIMENT:
    if not ENABLE_DRIVE:
        raise PermissionError("Set ENABLE_DRIVE=True")
    from google.colab import drive
    drive.mount("/content/drive")
    runtime_path=Path("/content/drive/MyDrive")/DRIVE_FOLDER/"experiments"/EXPERIMENT_ID/"workspace/.replication/controller.py"
    if not runtime_path.is_file():
        raise RuntimeError("Complete Version 3 notebook 00 first; its shared controller is missing")
    runtime_code=runtime_path.read_text(encoding="utf-8")
    if hashlib.sha256(runtime_code.encode()).hexdigest()!=CONTROLLER_SHA:
        raise ValueError("Shared runtime version differs; do not mix notebook packages")
    exec(compile(runtime_code,"<Colab shared runtime>","exec"),globals())
    rt=Replica(EXPERIMENT_ID,drive_base=DRIVE_FOLDER,allow_drive=True)
    if CLEAR_STALE_WRITER_LOCK:
        rt.clear_stale_lock(acknowledge_no_other_writer=CONFIRM_NO_OTHER_WRITER)
else:
    print("Bootstrap disabled: no Drive access or filesystem changes")


In [ ]:
if RUN_STAGE:
    if rt is None:
        raise RuntimeError("Enable Drive/bootstrap first")
    rt.require_stage('00')
    rt.install_sources(REGISTERED_FILES)
    rt.environment(allow_install=ALLOW_PACKAGE_INSTALL)
else:
    print("Stage execution disabled")


### Download missing versioned datasets

This action executes only with RUN_STAGE=True. Progress and worker resource measurements are written to the experiment run directory.

In [ ]:
if RUN_STAGE:
    for name in ("dataco","india_nepal","crisismmd"):
        manifest=rt.root/"data/manifests"/(name+".json")
        if manifest.exists():
            print("Reusing verified acquisition:",name)
            continue
        if not ALLOW_DATA_DOWNLOADS:
            raise PermissionError("Set ALLOW_DATA_DOWNLOADS=True to acquire the public datasets")
        rt.execute("01-download-"+name,["scripts/acquire_colab_datasets.py","--dataset",name],SYNC_INTERVAL_SECONDS)
else:
    print("Stage execution disabled")


## 6. Results and interpretation

A verified download establishes file provenance; it does not establish label quality or model accuracy.

The following cell reads this experiment outputs; definitions alone are not evidence that a stage ran.

In [ ]:
if rt is None:
    print("No attached experiment")
else:
    for path in sorted((rt.root/"data/manifests").glob("*.json")):
        record=read_json(path)
        print(path.name,record.get("file_count"),"files",record.get("total_bytes"),"bytes",record.get("published_license"))


## 7. Persist and hand over

Only after the preceding actions succeed, run the following cell to synchronize verified outputs and record this notebook completion. After an error, stop; repair/restart and rerun in order before recording completion. The next stage reads saved artifacts/components from Drive.

In [ ]:
if RUN_STAGE:
    rt.finish_stage('01')
else:
    print("Stage execution disabled")


## Recovery and boundaries

Run one writer per experiment. After interruption, disconnect and delete the Colab runtime, reconnect and restore this same experiment. Completed models are verified/reused; neural checkpoints resume compatible state. Unsynchronized work may repeat. Do not clear a writer lock until no other session is active.

No final-test fitting/scoring/tuning is performed. Model outputs do not verify incident truth or shipment exposure. Hosted Colab execution remains unverified until exercised in your session.